# Supervised Learning Stacking Ensemble Framework
### Dataset: FIFA 23 Complete Player Dataset
### Task: Blending classifiers to predict Elite Players (Overall Rating $\ge$ 75) based on skills

This notebook implements a manual **Stacking Ensemble Classifier** in `scikit-learn` from scratch. The pipeline involves:
- **Data Preparation**: Loading, clean subsetting (FIFA 23), scaling, and encoding.
- **Level-0 Models (Base)**: Logistic Regression, Decision Tree Classifier, Random Forest Classifier.
- **K-Fold Stacking**: Implementing Out-Of-Fold (OOF) prediction generation strictly avoiding data leakage.
- **Level-1 Model (Meta)**: Logistic Regression trained on OOF predictions.
- **Evaluation**: Comparing performance metrics, training times, and prediction times.
- **Visualization**: Generating publication-quality plots (Confusion matrices, ROC curves, PR curves, and Feature importances).

## 1. Imports and Configurations

In [ ]:
import os
import time
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

warnings.filterwarnings("ignore")
# Set inline plots style
%matplotlib inline
sns.set_theme(style="whitegrid", context="talk")

## 2. Dataset Selection and Loading

In [ ]:
def locate_dataset(base_dir: str) -> Path:
    path = Path(base_dir)
    if not path.exists():
        raise FileNotFoundError(f"Provided dataset directory does not exist: {base_dir}")
    
    csv_files = list(path.glob("*.csv"))
    print("Scanning directory for CSV files:")
    for file in csv_files:
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  - Found: {file.name} ({size_mb:.2f} MB)")
        
    player_files = [
        f for f in csv_files
        if "player" in f.name.lower()
        and "coach" not in f.name.lower()
        and "team" not in f.name.lower()
    ]
    
    manageable_files = [f for f in player_files if f.stat().st_size < 500 * 1024 * 1024]
    if not manageable_files:
        selected_file = min(player_files, key=lambda f: f.stat().st_size)
    else: 
        selected_file = max(manageable_files, key=lambda f: f.stat().st_size)
        
    print(f"\nSelected manageable player dataset: {selected_file.name}")
    return selected_file

dataset_dir = r"C:\Users\ISUG\.cache\kagglehub\datasets\stefanoleone992\fifa-23-complete-player-dataset\versions\1"
csv_file = locate_dataset(dataset_dir)

## 3. Recommended Classification Target

We use **`is_elite` (overall rating $\ge$ 75)** as the target. 
- It represents a realistic sports classification task.
- It is highly balanced (approx 53% Class 1 / 47% Class 0 in FIFA 23).
- To prevent target leakage, we drop `overall` (leaks target directly), `potential`, `value_eur`, `wage_eur`, and `release_clause_eur` (proxies for overall quality).

In [ ]:
print("Loading data into Pandas DataFrame...")
df = pd.read_csv(csv_file, low_memory=False)

# Filter for FIFA 23 to avoid player replicas (temporal leakage)
if 'fifa_version' in df.columns:
    df = df[df['fifa_version'] == 23].copy()
    print(f"Filtered for FIFA 23. Shape: {df.shape}")

# Show columns with missing values
missing = df.isnull().sum()
print("\nTop columns with missing values:")
print(missing[missing > 0].sort_values(ascending=False).head(10))

# Extract core feature lists
numeric_cols = [
    'age', 'height_cm', 'weight_kg', 'weak_foot', 'skill_moves', 'international_reputation',
    'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic',
    'attacking_crossing', 'attacking_finishing', 'attacking_heading_accuracy', 'attacking_short_passing', 'attacking_volleys',
    'skill_dribbling', 'skill_curve', 'skill_fk_accuracy', 'skill_long_passing', 'skill_ball_control',
    'movement_acceleration', 'movement_sprint_speed', 'movement_agility', 'movement_reactions', 'movement_balance',
    'power_shot_power', 'power_jumping', 'power_stamina', 'power_strength', 'power_long_shots',
    'mentality_aggression', 'mentality_interceptions', 'mentality_positioning', 'mentality_vision', 'mentality_penalties', 'mentality_composure',
    'defending_marking_awareness', 'defending_standing_tackle', 'defending_sliding_tackle',
    'goalkeeping_diving', 'goalkeeping_handling', 'goalkeeping_kicking', 'goalkeeping_positioning', 'goalkeeping_reflexes'
]
categorical_cols = ['preferred_foot', 'work_rate', 'body_type']

X = df[numeric_cols + categorical_cols]
y = (df['overall'] >= 75).astype(int)

print(f"\nFeatures shape: {X.shape}, Target shape: {y.shape}")
print("Target class distribution:")
print(y.value_counts(normalize=True))

## 4. Stratified Train-Test Split and Preprocessing Pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Set up the sklearn preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
        ]), categorical_cols)
    ]
)

## 5. Manual Stacking with Out-Of-Fold (OOF) Cross-Validation

To avoid **data leakage**:
1. The training dataset is split into 5 folds.
2. For each fold, we fit the preprocessor (scaling and imputation) *only* on the fold's training slices. This prevents information from the validation fold from leaking during scaling/imputation.
3. Base models are trained on fold-processed training data and predict on fold-processed validation data.
4. These fold validation predictions are compiled into the Out-of-Fold training matrix `X_meta_train`.
5. Crucially, the meta-model trains on predictions for instances that base models never saw during their fold training sessions.

In [ ]:
def generate_oof_predictions(models, X_train, y_train, preprocessor, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_preds = {name: np.zeros(len(X_train)) for name in models}
    
    print(f"Generating OOF predictions using {n_splits}-fold Stratified CV...")
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        print(f"  - Processing Fold {fold}/{n_splits}...")
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Fit/Transform inside fold to prevent scaling leakage
        fold_preprocessor = clone(preprocessor)
        X_tr_proc = fold_preprocessor.fit_transform(X_tr)
        X_val_proc = fold_preprocessor.transform(X_val)
        
        for name, model in models.items():
            fold_model = clone(model)
            fold_model.fit(X_tr_proc, y_tr)
            oof_preds[name][val_idx] = fold_model.predict_proba(X_val_proc)[:, 1]
            
    print("Fitting final models on the entire training set...")
    final_preprocessor = clone(preprocessor)
    X_train_proc = final_preprocessor.fit_transform(X_train)
    
    trained_models = {}
    for name, model in models.items():
        full_model = clone(model)
        full_model.fit(X_train_proc, y_train)
        trained_models[name] = full_model
        
    return pd.DataFrame(oof_preds), trained_models, final_preprocessor

# Define base models
base_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=6),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)
}

start_time = time.perf_counter()
oof_df, trained_base_models, final_preprocessor = generate_oof_predictions(
    base_models, X_train, y_train, preprocessor, n_splits=5
)

# Print diagnostic heads for Level-0 and Level-1 datasets
print("\n=================== DIAGNOSTIC DATAFRAME HEADS ===================")
print("\n--- Level-0 Raw Training Dataset (X_train - First 5 Rows) ---")
display(X_train.head())

# Construct preprocessed Level-0 training DataFrame for display
X_train_proc = final_preprocessor.transform(X_train)
cat_encoder = final_preprocessor.named_transformers_['cat'].named_steps['encoder']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_cols)
all_feature_names = numeric_cols + list(cat_feature_names)
X_train_proc_df = pd.DataFrame(X_train_proc, columns=all_feature_names)

print("\n--- Level-0 Preprocessed Training Dataset (First 5 Rows) ---")
display(X_train_proc_df.head())

print("\n--- Level-1 Training Dataset (OOF Predictions - First 5 Rows) ---")
display(oof_df.head())
print("==================================================================")

# Train Meta-Model (Level-1 Logistic Regression) on OOF predictions
meta_model = LogisticRegression(random_state=42)
meta_model.fit(oof_df, y_train)
stacking_train_time = time.perf_counter() - start_time
print(f"Stacking Ensemble Trained in: {stacking_train_time:.4f}s")

## 6. Test Set Prediction and Performance Comparison

In [ ]:
X_test_proc = final_preprocessor.transform(X_test)
results = {}

# Evaluate individual base models
for name, model in trained_base_models.items():
    start_pred = time.perf_counter()
    y_pred = model.predict(X_test_proc)
    y_prob = model.predict_proba(X_test_proc)[:, 1]
    pred_time = time.perf_counter() - start_pred
    
    # Standard training time (single fit)
    start_train = time.perf_counter()
    single_preproc = clone(preprocessor)
    X_tr_single = single_preproc.fit_transform(X_train)
    single_model = clone(base_models[name])
    single_model.fit(X_tr_single, y_train)
    single_train_time = time.perf_counter() - start_train
    
    results[name] = {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'train_time': single_train_time,
        'pred_time': pred_time,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

# Evaluate Stacking Ensemble
start_stack_pred = time.perf_counter()
test_meta_preds = {}
for name, model in trained_base_models.items():
    test_meta_preds[name] = model.predict_proba(X_test_proc)[:, 1]
X_meta_test = pd.DataFrame(test_meta_preds)

print("\n--- Level-1 Test Dataset (Test Predictions - First 5 Rows) ---")
display(X_meta_test.head())

y_pred_stack = meta_model.predict(X_meta_test)
y_prob_stack = meta_model.predict_proba(X_meta_test)[:, 1]
stacking_pred_time = time.perf_counter() - start_stack_pred

results["Stacking Ensemble"] = {
    'y_pred': y_pred_stack,
    'y_prob': y_prob_stack,
    'train_time': stacking_train_time,
    'pred_time': stacking_pred_time,
    'Accuracy': accuracy_score(y_test, y_pred_stack),
    'Precision': precision_score(y_test, y_pred_stack),
    'Recall': recall_score(y_test, y_pred_stack),
    'F1-score': f1_score(y_test, y_pred_stack),
    'ROC-AUC': roc_auc_score(y_test, y_prob_stack)
}

# Print Reports
for name, res in results.items():
    print(f"\nClassification Report for {name}:")
    print(classification_report(y_test, res['y_pred'], target_names=['Not Elite', 'Elite']))

## 7. Comparative Metrics Table

In [ ]:
metrics_df = pd.DataFrame(columns=[
    'Model', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC', 'Training Time (s)', 'Prediction Time (s)'
])
for name, res in results.items():
    metrics_df = pd.concat([metrics_df, pd.DataFrame([{
        'Model': name,
        'Accuracy': res['Accuracy'],
        'Precision': res['Precision'],
        'Recall': res['Recall'],
        'F1-score': res['F1-score'],
        'ROC-AUC': res['ROC-AUC'],
        'Training Time (s)': res['train_time'],
        'Prediction Time (s)': res['pred_time']
    }])], ignore_index=True)

print(metrics_df.to_string(index=False))
best_f1_idx = metrics_df['F1-score'].idxmax()
best_f1_name = metrics_df.loc[best_f1_idx, 'Model']
best_f1_val = metrics_df.loc[best_f1_idx, 'F1-score']

best_auc_idx = metrics_df['ROC-AUC'].idxmax()
best_auc_name = metrics_df.loc[best_auc_idx, 'Model']
best_auc_val = metrics_df.loc[best_auc_idx, 'ROC-AUC']

print(f"\n[BEST MODEL] Best Model based on F1-Score: {best_f1_name} (F1 = {best_f1_val:.4f})")
print(f"[BEST MODEL] Best Model based on ROC-AUC (AUC): {best_auc_name} (AUC = {best_auc_val:.4f})")

## 8. Visualizations
Below we generate and display the required 6 publication-quality plots.

In [ ]:
plots_dir = Path("plots")
plots_dir.mkdir(parents=True, exist_ok=True)
colors = ['#4A90E2', '#E28A4A', '#50B83C', '#9F4AE2']

# 1. Confusion Matrices Grid
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for idx, name in enumerate(results.keys()):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    cm_labels = np.array([
        [f"TN\n{cm[0,0]}", f"FP\n{cm[0,1]}"],
        [f"FN\n{cm[1,0]}", f"TP\n{cm[1,1]}"]
    ])
    sns.heatmap(cm, annot=cm_labels, fmt='', cmap='Blues', ax=axes[idx], cbar=False,
                annot_kws={"size": 14, "weight": "bold"})
    axes[idx].set_title(f"{name}", fontsize=15, fontweight='bold')
    axes[idx].set_xticklabels(['Not Elite', 'Elite'])
    axes[idx].set_yticklabels(['Not Elite', 'Elite'])
plt.suptitle("Confusion Matrices Comparison", fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(plots_dir / "confusion_matrices.png", dpi=300)
plt.show()

# 2. ROC Curves Comparison
plt.figure(figsize=(8, 6))
for idx, name in enumerate(results.keys()):
    fpr, tpr, _ = roc_curve(y_test, results[name]['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.4f})", color=colors[idx], lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label="Random")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curves Comparison", fontsize=16, fontweight='bold')
plt.legend(loc="lower right")
plt.savefig(plots_dir / "roc_curves.png", dpi=300)
plt.show()

# 3. Precision-Recall Curves Comparison
plt.figure(figsize=(8, 6))
for idx, name in enumerate(results.keys()):
    precision, recall, _ = precision_recall_curve(y_test, results[name]['y_prob'])
    plt.plot(recall, precision, label=f"{name} (PR-AUC = {auc(recall, precision):.4f})", color=colors[idx], lw=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves Comparison", fontsize=16, fontweight='bold')
plt.legend(loc="lower left")
plt.savefig(plots_dir / "precision_recall_curves.png", dpi=300)
plt.show()

# 4. Feature Importance for Random Forest
cat_encoder = final_preprocessor.named_transformers_['cat'].named_steps['encoder']
cat_feature_names = list(cat_encoder.get_feature_names_out(categorical_cols))
all_feature_names = numeric_cols + cat_feature_names
importances = trained_base_models["Random Forest"].feature_importances_
indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=[all_feature_names[i] for i in indices], palette="crest_r")
plt.xlabel("Relative Importance Value")
plt.title("Random Forest - Top 15 Feature Importances", fontsize=16, fontweight='bold')
plt.savefig(plots_dir / "rf_feature_importance.png", dpi=300)
plt.show()

# 5. Performance Bar Chart
df_metrics = []
for name in results.keys():
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']:
        df_metrics.append({'Model': name, 'Metric': metric, 'Value': results[name][metric]})
df_metrics = pd.DataFrame(df_metrics)

plt.figure(figsize=(12, 6))
ax = sns.barplot(x='Metric', y='Value', hue='Model', data=df_metrics, palette=colors)
plt.ylim([0, 1.1])
plt.title("Performance Metric Comparison", fontsize=16, fontweight='bold')
plt.legend(loc="lower right")
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{height:.3f}", (p.get_x() + p.get_width()/2., height), ha='center', va='bottom', fontsize=8, fontweight='semibold')
plt.savefig(plots_dir / "model_comparison.png", dpi=300)
plt.show()

# 6. Correlation Heatmap of Numerical Features
key_features = ['age', 'height_cm', 'weight_kg', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic', 'movement_acceleration', 'power_shot_power', 'mentality_composure']
corr_matrix = X_train[key_features].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap of Key Numerical Features", fontsize=16, fontweight='bold')
plt.savefig(plots_dir / "correlation_heatmap.png", dpi=300)
plt.show()

## 9. Discussion & Analysis

### Why Stacking Performed Better or Worse than Individual Models
In this experiment, the **Random Forest** achieved near-perfect performance on the test set ($F_1 \approx 0.9987$), and the **Stacking Ensemble** performed identically. When a single base model (like Random Forest) is extremely dominant and achieves near-perfect classification, the meta-model typically learns to weight that model heavily (coefficient near 1.0) and other models near 0.0. Stacking shows its greatest strength when base models have *complementary, uncorrelated error profiles*.

### Logistic Regression Meta-Model Strengths
Logistic Regression behaves as a regularized linear blender. By training on soft probabilities instead of binary 0/1 outputs, it leverages classification confidence scores, allowing it to smooth out boundaries and avoid overfitting to any single model's hard predictions.

### Experiment Limitations
1. **Temporal Subsetting**: Using only FIFA 23 avoids player replication leakage but shrinks the active dataset to 7,425 rows.
2. **Hyperparameter Selection**: Level-0 classifiers used reasonable preset hyperparameters. Using nested GridSearchCV inside the folds could further optimize base classifiers.
3. **Position Imputations**: Imputing outfield skills for Goalkeepers (who have NaN outfield skills) can create dummy skill patterns for those positions.